# MovieLens-1M Basic Data Inspection

**Project:** Personalization and Recommendation Under Privacy Constraints  
**Course:** DATA 298A  
**Team:** Team 5  

## Purpose

This notebook performs an initial inspection of the raw MovieLens-1M dataset before preprocessing.

The inspection covers:

- Dataset schema
- Number of records
- Number of unique users and items
- Timestamp range
- Missing values
- Duplicate records
- Basic interaction statistics

No preprocessing or train/validation/test splitting is performed in this notebook.
The raw dataset remains unchanged.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

# -----------------------------
# Paths
# -----------------------------

RAW_DIR = Path(
    "/content/drive/MyDrive/298A/"
    "04_Data/02_Raw_Data/MovieLens-1M/ml-1m"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/298A/"
    "04_Data/03_Data_Inspection/MovieLens-1M"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data directory:")
print(RAW_DIR)

print("\nInspection output directory:")
print(OUTPUT_DIR)

Raw data directory:
/content/drive/MyDrive/298A/04_Data/02_Raw_Data/MovieLens-1M/ml-1m

Inspection output directory:
/content/drive/MyDrive/298A/04_Data/03_Data_Inspection/MovieLens-1M


## Load raw MovieLens-1M files

In [3]:
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding="latin-1"
)

users = pd.read_csv(
    RAW_DIR / "users.dat",
    sep="::",
    engine="python",
    names=[
        "user_id",
        "gender",
        "age",
        "occupation",
        "zip_code"
    ],
    encoding="latin-1"
)

movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=[
        "movie_id",
        "title",
        "genres"
    ],
    encoding="latin-1"
)

print("Datasets loaded successfully.")

Datasets loaded successfully.


## Preview raw data

In [4]:
print("RATINGS")
display(ratings.head())

print("\nUSERS")
display(users.head())

print("\nMOVIES")
display(movies.head())

RATINGS


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291



USERS


,user_id,gender,age,occupation,zip_code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455



MOVIES


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
print("\nRatings schema:")
ratings.info()

print("\nUsers schema:")
users.info()

print("\nMovies schema:")
movies.info()


Ratings schema:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   user_id    1000209 non-null  int64
 1   movie_id   1000209 non-null  int64
 2   rating     1000209 non-null  int64
 3   timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB

Users schema:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     6040 non-null   int64 
 1   gender      6040 non-null   object
 2   age         6040 non-null   int64 
 3   occupation  6040 non-null   int64 
 4   zip_code    6040 non-null   object
dtypes: int64(3), object(2)
memory usage: 236.1+ KB

Movies schema:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column    No

## Basic dataset statistics

In [6]:
summary = {
    "ratings_rows": len(ratings),
    "users_rows": len(users),
    "movies_rows": len(movies),

    "unique_users_in_ratings": ratings["user_id"].nunique(),
    "unique_movies_in_ratings": ratings["movie_id"].nunique(),

    "min_rating": ratings["rating"].min(),
    "max_rating": ratings["rating"].max(),
    "mean_rating": ratings["rating"].mean()
}

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["metric", "value"]
)

display(summary_df)

,metric,value
0,ratings_rows,1.000209e+06
1,users_rows,6.040000e+03
2,movies_rows,3.883000e+03
3,unique_users_in_ratings,6.040000e+03
4,unique_movies_in_ratings,3.706000e+03
5,min_rating,1.000000e+00
6,max_rating,5.000000e+00
7,mean_rating,3.581564e+00


## Timestamp inspection

In [7]:
ratings["datetime"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s"
)

timestamp_summary = pd.DataFrame({
    "metric": [
        "earliest_interaction",
        "latest_interaction"
    ],
    "value": [
        ratings["datetime"].min(),
        ratings["datetime"].max()
    ]
})

display(timestamp_summary)

,metric,value
0,earliest_interaction,2000-04-25 23:05:32
1,latest_interaction,2003-02-28 17:49:50


In [8]:
ratings[
    ["user_id", "movie_id", "rating", "timestamp", "datetime"]
].head()

,user_id,movie_id,rating,timestamp,datetime
0,1,1193,5,978300760,2000-12-31 22:12:40
1,1,661,3,978302109,2000-12-31 22:35:09
2,1,914,3,978301968,2000-12-31 22:32:48
3,1,3408,4,978300275,2000-12-31 22:04:35
4,1,2355,5,978824291,2001-01-06 23:38:11


## Missing value inspection

In [9]:
missing_summary = pd.DataFrame({
    "ratings": ratings.isnull().sum(),
}).dropna(how="all")

print("Ratings missing values:")
display(ratings.isnull().sum().to_frame("missing_count"))

print("\nUsers missing values:")
display(users.isnull().sum().to_frame("missing_count"))

print("\nMovies missing values:")
display(movies.isnull().sum().to_frame("missing_count"))

Ratings missing values:


,missing_count
user_id,0
movie_id,0
rating,0
timestamp,0
datetime,0



Users missing values:


,missing_count
user_id,0
gender,0
age,0
occupation,0
zip_code,0



Movies missing values:


,missing_count
movie_id,0
title,0
genres,0


## Duplicate inspection

In [10]:
exact_duplicate_ratings = ratings.duplicated(
    subset=["user_id", "movie_id", "rating", "timestamp"]
).sum()

duplicate_user_movie_pairs = ratings.duplicated(
    subset=["user_id", "movie_id"],
    keep=False
).sum()

print(
    "Exact duplicate rating rows:",
    exact_duplicate_ratings
)

print(
    "Rows belonging to repeated user-movie pairs:",
    duplicate_user_movie_pairs
)

Exact duplicate rating rows: 0
Rows belonging to repeated user-movie pairs: 0


In [11]:
print(
    "Duplicate user IDs:",
    users["user_id"].duplicated().sum()
)

print(
    "Duplicate movie IDs:",
    movies["movie_id"].duplicated().sum()
)

Duplicate user IDs: 0
Duplicate movie IDs: 0


## User interaction statistics

In [12]:
user_interaction_counts = (
    ratings
    .groupby("user_id")
    .size()
    .rename("interaction_count")
)

interaction_stats = (
    user_interaction_counts
    .describe()
    .to_frame()
)

display(interaction_stats)

,interaction_count
count,6040.000000
mean,165.597517
std,192.747029
min,20.000000
25%,44.000000
50%,96.000000
75%,208.000000
max,2314.000000


In [13]:
print(
    "Minimum interactions per user:",
    user_interaction_counts.min()
)

print(
    "Median interactions per user:",
    user_interaction_counts.median()
)

print(
    "Mean interactions per user:",
    round(user_interaction_counts.mean(), 2)
)

print(
    "Maximum interactions per user:",
    user_interaction_counts.max()
)

Minimum interactions per user: 20
Median interactions per user: 96.0
Mean interactions per user: 165.6
Maximum interactions per user: 2314


## Basic validation checks

In [14]:
checks = {
    "Ratings table is not empty":
        len(ratings) > 0,

    "Users table is not empty":
        len(users) > 0,

    "Movies table is not empty":
        len(movies) > 0,

    "All rating users exist in users.dat":
        set(ratings["user_id"]).issubset(
            set(users["user_id"])
        ),

    "All rated movies exist in movies.dat":
        set(ratings["movie_id"]).issubset(
            set(movies["movie_id"])
        ),

    "Ratings are between 1 and 5":
        ratings["rating"].between(1, 5).all(),

    "Timestamps are present":
        ratings["timestamp"].notna().all()
}

validation_df = pd.DataFrame(
    checks.items(),
    columns=["check", "passed"]
)

display(validation_df)

,check,passed
0,Ratings table is not empty,True
1,Users table is not empty,True
2,Movies table is not empty,True
3,All rating users exist in users.dat,True
4,All rated movies exist in movies.dat,True
5,Ratings are between 1 and 5,True
6,Timestamps are present,True


In [15]:
assert validation_df["passed"].all(), \
    "One or more validation checks failed."

print("All basic validation checks passed.")

All basic validation checks passed.


In [16]:
# -----------------------------
# Save inspection artifacts
# -----------------------------

summary_df.to_csv(
    OUTPUT_DIR / "dataset_summary.csv",
    index=False
)

timestamp_summary.to_csv(
    OUTPUT_DIR / "timestamp_summary.csv",
    index=False
)

interaction_stats.to_csv(
    OUTPUT_DIR / "user_interaction_statistics.csv"
)

validation_df.to_csv(
    OUTPUT_DIR / "validation_checks.csv",
    index=False
)

print("Inspection artifacts saved to:")
print(OUTPUT_DIR)

Inspection artifacts saved to:
/content/drive/MyDrive/298A/04_Data/03_Data_Inspection/MovieLens-1M


## Initial Inspection Summary

The raw MovieLens-1M dataset was successfully loaded and inspected.

The inspection confirmed that:

- The dataset contains approximately one million rating interactions.
- User and movie identifiers are available for sequential recommendation.
- Interaction timestamps are available for chronological ordering.
- Basic missing-value and duplicate checks were completed.
- User interaction counts were inspected to understand sequence lengths.
- Basic validation checks passed.

No filtering, preprocessing, or train/validation/test splitting was performed at this stage.

The next step is to reproduce the preprocessing protocol required by SASRec while keeping the raw dataset unchanged.